In [1]:
"""
Project:Hack 27 - Challenge 4 (Human-Centric Data)
Team Pulse & Performance - per-team dashboards (team-facing view)
==================================================================

Generates ONE HTML dashboard per team, matching the
"Team Pulse & Performance" mock-up, using THREE surveys (S1, S2, S3)
collected across the hack.

    +-------------------------------------------------------+
    |             Team Pulse & Performance                  |
    +-------------------------------------------------------+
    |  [Wellbeing emoji]   [Team Access 58]  [Cog. Load 82] |
    +-----------------+---------------------+---------------+
    | Team Strengths  |   Team Mood Tracker |  How Strong   |
    | & Gaps          |   How We're Feeling |   Is Our      |
    |  We're Great At |   Our Stress Level  |  Solution?    |
    |  We Can Improve |                     |   (gauge)     |
    +-----------------+---------------------+---------------+
    |                     Take Action!                      |
    | [Strengthen Teamwork] [Ease the Load] [Boost Solution]|
    +-------------------------------------------------------+

Rater ID format
---------------
IDs are stored as negative integers (e.g. -1700002). Take abs(), pad to 7
digits; first 2 chars -> Team, last char -> Individual.

What's different from the S1/S2 version
---------------------------------------
* Loads Survey 3 as well.
* "Latest" readings come from S3 (with S2 -> S1 fallback).
* Trend arrows compare S3 vs S2 (so they show the most recent movement).
* Solution gauge now averages up to 5 challenge questions (S3 has two extras:
  "commercial application" and "beneficial").
* Mood / stress tracker plots a full S1 -> S2 -> S3 trajectory per team.

Usage
-----
    python team_pulse_dashboards.py

Outputs
-------
    ./output/team_<NN>_pulse.html    one per team
    ./output/team_pulse_summary.csv  tidy roll-up
"""

from __future__ import annotations

import re
import time
from pathlib import Path

import numpy as np
import pandas as pd

# --------------------------------------------------------------------------- #
# Config
# --------------------------------------------------------------------------- #

# Put the three CSVs next to this script (or adjust these paths).
SURVEY1 = Path("Survey_1_Raw_Data_By_Rater_.csv")
SURVEY2 = Path("Survey_2_Raw_Data_By_Rater_.csv")
SURVEY3 = Path("Survey_3_Raw_Data_By_Rater_.csv")
SURVEY4 = Path("Survey_4_Raw_Data_By_Rater_.csv")

OUTPUT = Path("./output")
OUTPUT.mkdir(parents=True, exist_ok=True)

# Thresholds (0-100 scale)
GOOD = 70    # >= green / "Good" / "Strong"
OK = 50      # >= amber, below -> red

# ---- AI Summary (Ollama) ---------------------------------------------------
# Uses the official `ollama` Python package. Install with:
#     pip install ollama
# Make sure the Ollama service is running locally (`ollama serve`) and that
# the model below has been pulled (`ollama pull llama3.2:1b`).
# If the package isn't installed or the service is unreachable, the script
# falls back to the rule-based summary - no crash.
OLLAMA_MODEL = "llama3"
AI_SUMMARY_ENABLED = True     # set False to always skip the LLM call


# --------------------------------------------------------------------------- #
# Data loading
# --------------------------------------------------------------------------- #

def _clean(label: str) -> str:
    if not isinstance(label, str):
        return ""
    return re.sub(r"<[^>]+>", "", label).replace("&nbsp;", " ").strip()


def load_survey(path: Path) -> pd.DataFrame:
    """Load a By-Rater CSV that has a two-row header."""
    df = pd.read_csv(path, header=[0, 1])
    flat = []
    for grp, sub in df.columns:
        g = "" if str(grp).startswith("Unnamed") else str(grp)
        s = "" if str(sub).startswith("Unnamed") else str(sub)
        flat.append(_clean(s or g))
    df.columns = flat
    df = df.loc[:, [c for c in df.columns if c]]
    df = df.dropna(subset=["Rater"]).copy()
    df["Rater"] = df["Rater"].astype(int)
    return df


def add_team_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Parse |Rater| zero-padded to 7 chars: first 2 = Team, last = Individual."""
    ids = df["Rater"].abs().astype(int).astype(str).str.zfill(7)
    df = df.copy()
    df["Team"] = ids.str[:2]
    df["Individual"] = ids.str[-1]
    return df


# --------------------------------------------------------------------------- #
# Metric extraction
# --------------------------------------------------------------------------- #

def _find(df: pd.DataFrame, needle: str) -> str | None:
    n = needle.lower()
    for c in df.columns:
        if n in c.lower():
            return c
    return None


# Maps canonical metric names to the substring we look for in survey headers.
METRIC_NEEDLES = {
    "Wellbeing":     "wellbeing",
    "Access":        "team access",
    "Productivity":  "productivity",
    "Capacity":      "capacity",
    "Capability":    "capability",
    "Collaboration": "collaboration",
    "Safety":        "safely",
    "CogLoad":       "cognitive",
}

# Challenge-related metrics for the solution gauge.
# S2 had three; S3 adds two more ("commercial application" and "beneficial");
# S4 adds "presented". We look for each needle; any that aren't present in a
# survey are simply ignored.
CHALLENGE_NEEDLES = {
    "Solution":    "solution meets",
    "Innovation":  "innovative",
    "Ambition":    "ambitious",
    "Commercial":  "commercial",
    "Beneficial":  "beneficial",
    "Presented":   "presented",
}


def team_means(df: pd.DataFrame, needles: dict) -> pd.DataFrame:
    """Return a DataFrame indexed by Team with one column per canonical metric."""
    cols = {}
    for canon, n in needles.items():
        found = _find(df, n)
        if found is not None:
            cols[found] = canon
    if not cols:
        return pd.DataFrame()

    sub = df[["Team"] + list(cols.keys())].rename(columns=cols)
    for c in cols.values():
        sub[c] = pd.to_numeric(sub[c], errors="coerce")
        sub.loc[sub[c] < 0, c] = np.nan   # -1 is the survey's missing sentinel
    return sub.groupby("Team").mean(numeric_only=True)


def individual_series(df: pd.DataFrame, needles: dict) -> pd.DataFrame:
    """Per-individual (not aggregated) values, used to draw the mood tracker."""
    cols = {}
    for canon, n in needles.items():
        found = _find(df, n)
        if found is not None:
            cols[found] = canon
    if not cols:
        return pd.DataFrame()
    sub = df[["Team", "Individual"] + list(cols.keys())].rename(columns=cols)
    for c in cols.values():
        sub[c] = pd.to_numeric(sub[c], errors="coerce")
        sub.loc[sub[c] < 0, c] = np.nan
    return sub


# --------------------------------------------------------------------------- #
# Derived signals
# --------------------------------------------------------------------------- #

def mood_label(v: float) -> tuple[str, str, str]:
    """Return (face_colour, label, emoji_svg_fragment)."""
    if pd.isna(v):
        return ("#94a3b8", "No data", _face("#94a3b8", mouth="flat"))
    if v >= GOOD:
        return ("#22c55e", "Good", _face("#22c55e", mouth="smile"))
    if v >= OK:
        return ("#f59e0b", "OK", _face("#f59e0b", mouth="flat"))
    return ("#ef4444", "Low", _face("#ef4444", mouth="frown"))


def load_label(v: float) -> tuple[str, str, str]:
    """Cognitive load interpreter - high is BAD here."""
    if pd.isna(v):
        return ("#94a3b8", "No data", _face("#94a3b8", mouth="flat"))
    if v <= 40:
        return ("#22c55e", "Light", _face("#22c55e", mouth="smile"))
    if v <= 65:
        return ("#f59e0b", "Manageable", _face("#f59e0b", mouth="flat"))
    return ("#ef4444", "Overloaded", _face("#ef4444", mouth="frown"))


def access_label(v: float) -> tuple[str, str, str]:
    return mood_label(v)   # same 70/50 thresholds read correctly


def _face(colour: str, mouth: str = "smile") -> str:
    """Return an inline SVG emoji-style face."""
    if mouth == "smile":
        mouth_path = "M 22 38 Q 32 48 42 38"
    elif mouth == "frown":
        mouth_path = "M 22 42 Q 32 32 42 42"
    else:
        mouth_path = "M 22 40 L 42 40"
    return (
        f"<svg viewBox='0 0 64 64' width='48' height='48'>"
        f"<circle cx='32' cy='32' r='28' fill='{colour}'/>"
        f"<circle cx='24' cy='26' r='3' fill='#fff'/>"
        f"<circle cx='40' cy='26' r='3' fill='#fff'/>"
        f"<path d='{mouth_path}' stroke='#fff' stroke-width='3' "
        f"fill='none' stroke-linecap='round'/>"
        f"</svg>"
    )


def strengths_and_gaps(row: pd.Series) -> tuple[list[str], list[str]]:
    """Classify each metric as a strength (>=GOOD) or a gap (<OK)."""
    friendly = {
        "Wellbeing":     "Wellbeing",
        "Access":        "Team Access",
        "Productivity":  "Productivity",
        "Capacity":      "Capacity",
        "Capability":    "Skills & Capability",
        "Collaboration": "Collaboration",
        "Safety":        "Psychological Safety",
        "CogLoad":       "Managing Workload",
    }
    strengths, gaps = [], []
    for metric, label in friendly.items():
        v = row.get(metric)
        if pd.isna(v):
            continue
        # Cognitive load is inverted - low load is a strength.
        if metric == "CogLoad":
            score = 100 - v
        else:
            score = v
        if score >= GOOD:
            strengths.append(label)
        elif score < OK:
            gaps.append(label)
    return strengths[:4], gaps[:4]


# --------------------------------------------------------------------------- #
# DISC-style team-dynamics analysis (Survey 1 only)
# --------------------------------------------------------------------------- #
#
# Caveat: this is a TWO-AXIS approximation, not a full DISC assessment.
# We only have two sliders per person:
#   * Personal Style            -> task-focused (<50) vs people-focused (>=50)
#   * Preferred Working Culture -> structured/cautious (<50) vs fast-paced (>=50)
#
# Mapping those two axes onto the classic DISC quadrant grid:
#              LOW pace (structured) | HIGH pace (fast)
#   TASK       C - Conscientious     | D - Dominance
#   PEOPLE     S - Steadiness        | I - Influence

DISC_META = {
    "D": {"name": "Dominance",      "colour": "#ef4444",
          "blurb": "Drivers - decisive, direct, results-first."},
    "I": {"name": "Influence",      "colour": "#f59e0b",
          "blurb": "Energisers - sociable, persuasive, idea-rich."},
    "S": {"name": "Steadiness",     "colour": "#22c55e",
          "blurb": "Anchors - supportive, patient, team-keepers."},
    "C": {"name": "Conscientious",  "colour": "#3b82f6",
          "blurb": "Analysts - precise, careful, quality-driven."},
}


def classify_disc(personal_style: float,
                  preferred_culture: float) -> str | None:
    """Return one of 'D' / 'I' / 'S' / 'C' from the two slider values."""
    if pd.isna(personal_style) or pd.isna(preferred_culture):
        return None
    people = personal_style >= 50          # high = people-focused
    fast = preferred_culture >= 50         # high = fast-paced
    if not people and not fast:
        return "C"
    if not people and fast:
        return "D"
    if people and not fast:
        return "S"
    return "I"


def team_disc_profile(s1: pd.DataFrame, team: str) -> dict:
    """Per-team DISC mix plus diversity + commentary."""
    sub = s1[s1["Team"] == team].copy()
    if sub.empty:
        return {"counts": {"D": 0, "I": 0, "S": 0, "C": 0},
                "percents": {"D": 0, "I": 0, "S": 0, "C": 0},
                "dominant": None, "n": 0, "diversity": 0.0,
                "commentary": "No Survey 1 data for this team."}

    # Clean the two inputs (-1 = missing sentinel)
    ps = pd.to_numeric(sub["Personal Style"], errors="coerce")
    pc = pd.to_numeric(sub["Preferred Working Culture"], errors="coerce")
    ps = ps.where(ps >= 0, np.nan)
    pc = pc.where(pc >= 0, np.nan)

    labels = [classify_disc(a, b) for a, b in zip(ps, pc)]
    labels = [lbl for lbl in labels if lbl]
    n = len(labels)

    counts = {k: labels.count(k) for k in "DISC"}
    percents = {k: (100 * counts[k] / n) if n else 0 for k in "DISC"}
    dominant = max(counts, key=counts.get) if n else None

    # Shannon-style diversity, normalised 0-1. 1 = perfectly balanced mix.
    if n == 0:
        diversity = 0.0
    else:
        import math
        probs = [counts[k] / n for k in "DISC" if counts[k] > 0]
        entropy = -sum(p * math.log(p) for p in probs)
        diversity = max(0.0, entropy / math.log(4))   # clamp; log(1)=0 -> -0

    commentary = _disc_commentary(counts, percents, dominant, diversity, n)

    return {"counts": counts, "percents": percents, "dominant": dominant,
            "n": n, "diversity": diversity, "commentary": commentary}


def _disc_commentary(counts: dict, percents: dict,
                     dominant: str | None, diversity: float, n: int) -> str:
    """Plain-English read of the team's DISC mix."""
    if n == 0 or dominant is None:
        return "Not enough responses to read the team's dynamic."

    parts: list[str] = []

    # Opening sentence: dominant style + how pronounced it is.
    dom_share = percents[dominant]
    dom_name = DISC_META[dominant]["name"]
    if dom_share >= 60:
        parts.append(
            f"This team is heavily {dom_name}-led "
            f"({dom_share:.0f}% of respondents)."
        )
    elif dom_share >= 40:
        parts.append(
            f"The team leans {dom_name} "
            f"({dom_share:.0f}%) without being one-note."
        )
    else:
        parts.append(
            f"No single style dominates; {dom_name} is the plurality "
            f"at {dom_share:.0f}%."
        )

    # Diversity read.
    if diversity >= 0.9:
        parts.append("All four styles are represented - a well-balanced mix.")
    elif diversity >= 0.7:
        parts.append("Good spread across styles with healthy contrast.")
    elif diversity >= 0.4:
        parts.append("Two styles carry most of the team; watch for blind spots.")
    else:
        parts.append("The team is stylistically narrow - diversity is low.")

    # Specific tensions / gaps worth flagging.
    missing = [DISC_META[k]["name"] for k in "DISC" if counts[k] == 0]
    if missing and n >= 3:
        parts.append(f"No one is scoring as {' or '.join(missing)}.")

    # Risk flags based on common friction patterns. These can stack - it's
    # fine for a team to trigger more than one.
    d_share = percents["D"]
    s_share = percents["S"]
    c_share = percents["C"]
    i_share = percents["I"]

    if d_share >= 40 and s_share + c_share < 35:
        parts.append(
            "Risk: lots of drive, little caution - decisions may outrun "
            "the detail. Bring in a 'devil's advocate' role."
        )
    if c_share >= 40 and d_share + i_share < 35:
        parts.append(
            "Risk: rigour is strong but momentum may stall. "
            "Time-box decisions to avoid analysis paralysis."
        )
    if i_share >= 40 and c_share < 25:
        parts.append(
            "Risk: ideas flow freely but follow-through may slip. "
            "Assign one person to own delivery tracking."
        )
    if s_share >= 40 and d_share < 20:
        parts.append(
            "Risk: the team is harmonious but may avoid hard calls. "
            "Rotate a 'challenger' role each stand-up."
        )

    return " ".join(parts)


def solution_score(row: pd.Series) -> float:
    """Mean of whichever challenge questions are available in the row."""
    vals = []
    for m in ["Solution", "Innovation", "Ambition",
              "Commercial", "Beneficial", "Presented"]:
        v = row.get(m)
        if pd.notna(v):
            vals.append(v)
    return float(np.mean(vals)) if vals else np.nan


def actions_for(row: pd.Series, sol: float) -> dict[str, list[str]]:
    """Three action buckets: Strengthen Teamwork / Ease the Load / Boost Solution."""
    team_actions, load_actions, sol_actions = [], [], []

    # Teamwork bucket - triggered by safety or collaboration weakness
    safety = row.get("Safety", np.nan)
    collab = row.get("Collaboration", np.nan)
    if pd.notna(safety) and safety < OK:
        team_actions.append("Run a psychological safety check-in.")
    if pd.notna(collab) and collab < OK:
        team_actions.append("Improve our communication style.")
    if pd.notna(collab) and OK <= collab < GOOD:
        team_actions.append("Agree a quick sync cadence.")
    if not team_actions:
        team_actions.append("Keep current rhythm - it's working.")

    # Load bucket - triggered by cognitive load / capacity pressure
    cog = row.get("CogLoad", np.nan)
    cap = row.get("Capacity", np.nan)
    if pd.notna(cog) and cog > 70:
        load_actions.append("Drop low-priority tasks.")
        load_actions.append("Timebox to what must ship.")
    elif pd.notna(cog) and cog > 55:
        load_actions.append("Trim nice-to-haves from scope.")
    if pd.notna(cap) and cap < OK:
        load_actions.append("Share workload across the team.")
    if not load_actions:
        load_actions.append("Load is healthy - keep pace steady.")

    # Solution bucket - triggered by challenge-question score
    if pd.isna(sol):
        sol_actions.append("Add a check-in on solution fit.")
    elif sol < OK:
        sol_actions.append("Test our idea with a user.")
        sol_actions.append("Re-read the challenge brief.")
    elif sol < GOOD:
        sol_actions.append("Pressure-test our demo story.")
    else:
        sol_actions.append("Sharpen the pitch - strong base already.")

    return {
        "Strengthen Teamwork": team_actions[:2],
        "Ease the Load":       load_actions[:2],
        "Boost Our Solution":  sol_actions[:2],
    }


# --------------------------------------------------------------------------- #
# SVG generators - mood line, stress line, gauge
# --------------------------------------------------------------------------- #

def bar_chart_svg(values: list[float], stroke: str, fill: str,
                  labels=("Survey 1", "Survey 2", "Survey 3"),
                  inverted: bool = False) -> str:
    """Three-bar chart - one bar per survey. `values` are already aggregated
    team means (length 3). NaN -> greyed-out bar.

    If `inverted=True` (e.g. cognitive load, where high = bad), bars are
    coloured by threshold band: green <=40, amber <=65, red >65. This makes
    'taller = worse' visually obvious instead of misleading.
    """
    W, H = 320, 130
    pad_l, pad_r, pad_t, pad_b = 36, 16, 16, 34
    inner_w, inner_h = W - pad_l - pad_r, H - pad_t - pad_b

    n = len(values)
    # Bars take 60% of the slot, gaps take 40%
    slot_w = inner_w / n
    bar_w = slot_w * 0.55

    axis_y = pad_t + inner_h

    # Y-axis gridlines at 0 / 50 / 100
    grid = ""
    for v in (0, 50, 100):
        y = pad_t + inner_h * (1 - v / 100)
        grid += (
            f"<line x1='{pad_l}' y1='{y:.1f}' x2='{W - pad_r}' y2='{y:.1f}' "
            f"stroke='#e2e8f0' stroke-dasharray='2 3'/>"
            f"<text x='{pad_l - 4}' y='{y + 3:.1f}' text-anchor='end' "
            f"font-size='9' fill='#94a3b8'>{v}</text>"
        )

    def _bar_colours(v: float) -> tuple[str, str]:
        """Return (fill, stroke) for a bar, per-bar for inverted metrics."""
        if not inverted:
            return fill, stroke
        # Cognitive-load thresholds: low is good
        if v <= 40:
            return "#86efac", "#16a34a"
        if v <= 65:
            return "#fdba74", "#ea580c"
        return "#fca5a5", "#dc2626"

    bars = ""
    value_labels = ""
    x_labels = ""
    for i, v in enumerate(values):
        cx = pad_l + slot_w * (i + 0.5)
        bx = cx - bar_w / 2

        if pd.isna(v):
            # Greyed-out placeholder bar
            by = axis_y - 4
            bh = 4
            bars += (
                f"<rect x='{bx:.1f}' y='{by:.1f}' width='{bar_w:.1f}' "
                f"height='{bh:.1f}' fill='#cbd5e1' rx='2'/>"
            )
            value_labels += (
                f"<text x='{cx:.1f}' y='{axis_y - 10:.1f}' "
                f"text-anchor='middle' font-size='10' fill='#94a3b8'>-</text>"
            )
        else:
            bh = inner_h * (v / 100)
            by = axis_y - bh
            bar_fill, bar_stroke = _bar_colours(v)
            bars += (
                f"<rect x='{bx:.1f}' y='{by:.1f}' width='{bar_w:.1f}' "
                f"height='{bh:.1f}' fill='{bar_fill}' stroke='{bar_stroke}' "
                f"stroke-width='1.5' rx='2'/>"
            )
            value_labels += (
                f"<text x='{cx:.1f}' y='{by - 4:.1f}' text-anchor='middle' "
                f"font-size='11' font-weight='600' fill='{bar_stroke}'>"
                f"{v:.0f}</text>"
            )

        x_labels += (
            f"<text x='{cx:.1f}' y='{H - 10}' text-anchor='middle' "
            f"font-size='10' fill='#64748b'>{labels[i]}</text>"
        )

    return f"""
    <svg viewBox='0 0 {W} {H}' width='100%' preserveAspectRatio='xMidYMid meet'>
      {grid}
      <line x1='{pad_l}' y1='{axis_y}' x2='{W - pad_r}' y2='{axis_y}'
            stroke='#cbd5e1' stroke-width='1.2'/>
      {bars}
      {value_labels}
      {x_labels}
    </svg>
    """


def line_chart_svg(points: list[float], stroke: str, fill: str,
                   labels=("Start", "Now", "Last")) -> str:
    """Legacy line chart - kept for reference but no longer used."""
    W, H = 320, 120
    pad_l, pad_r, pad_t, pad_b = 30, 20, 15, 28
    inner_w, inner_h = W - pad_l - pad_r, H - pad_t - pad_b

    # Replace NaN with the series mean so we can still draw something
    clean = [p for p in points if not (isinstance(p, float) and np.isnan(p))]
    if not clean:
        return (f"<svg viewBox='0 0 {W} {H}' width='100%'>"
                f"<text x='{W/2}' y='{H/2}' text-anchor='middle' "
                f"font-size='12' fill='#64748b'>No data</text></svg>")
    mean = float(np.mean(clean))
    vals = [mean if (isinstance(p, float) and np.isnan(p)) else p for p in points]

    n = len(vals)
    xs = [pad_l + (inner_w * i / max(1, n - 1)) for i in range(n)]
    # Map 0-100 to y (100 at top, 0 at bottom)
    ys = [pad_t + inner_h * (1 - v / 100) for v in vals]

    # Line path
    line_path = " ".join(
        f"{'M' if i == 0 else 'L'} {x:.1f} {y:.1f}"
        for i, (x, y) in enumerate(zip(xs, ys))
    )
    # Filled area (drop to baseline, back to start)
    area_path = (
        f"M {xs[0]:.1f} {pad_t + inner_h:.1f} " +
        " ".join(f"L {x:.1f} {y:.1f}" for x, y in zip(xs, ys)) +
        f" L {xs[-1]:.1f} {pad_t + inner_h:.1f} Z"
    )

    # Axis + dots
    dots = "".join(
        f"<circle cx='{x:.1f}' cy='{y:.1f}' r='3.5' fill='{stroke}'/>"
        for x, y in zip(xs, ys)
    )
    # Three labels (Start / Now / Last) evenly spaced under the chart
    label_xs = [pad_l, pad_l + inner_w / 2, pad_l + inner_w]
    lbl_txt = "".join(
        f"<text x='{x:.1f}' y='{H - 8}' text-anchor='middle' "
        f"font-size='10' fill='#64748b'>{t}</text>"
        for x, t in zip(label_xs, labels)
    )

    axis_y = pad_t + inner_h
    return f"""
    <svg viewBox='0 0 {W} {H}' width='100%' preserveAspectRatio='xMidYMid meet'>
      <line x1='{pad_l}' y1='{axis_y}' x2='{W - pad_r}' y2='{axis_y}'
            stroke='#cbd5e1'/>
      <path d='{area_path}' fill='{fill}' opacity='0.35'/>
      <path d='{line_path}' stroke='{stroke}' stroke-width='2' fill='none'
            stroke-linecap='round' stroke-linejoin='round'/>
      {dots}
      {lbl_txt}
    </svg>
    """


def gauge_svg(value: float) -> tuple[str, str, str]:
    """Semi-circular gauge from 0 (red) to 100 (green). Returns (svg, label, col)."""
    if pd.isna(value):
        value_display = 0
        label = "No data"
        col = "#94a3b8"
    elif value >= GOOD:
        value_display = value
        label = "Strong & Innovative"
        col = "#22c55e"
    elif value >= OK:
        value_display = value
        label = "On Track"
        col = "#f59e0b"
    else:
        value_display = value
        label = "Needs a Push"
        col = "#ef4444"

    import math
    cx, cy, r = 120, 110, 80

    def arc(start_deg, end_deg, colour):
        x1 = cx + r * math.cos(math.radians(180 - start_deg))
        y1 = cy - r * math.sin(math.radians(180 - start_deg))
        x2 = cx + r * math.cos(math.radians(180 - end_deg))
        y2 = cy - r * math.sin(math.radians(180 - end_deg))
        large = 1 if (end_deg - start_deg) > 180 else 0
        return (
            f"<path d='M {x1:.1f} {y1:.1f} A {r} {r} 0 {large} 1 "
            f"{x2:.1f} {y2:.1f}' stroke='{colour}' stroke-width='18' "
            f"fill='none' stroke-linecap='butt'/>"
        )

    bands = arc(0, 60, "#ef4444") + arc(60, 120, "#f59e0b") + arc(120, 180, "#22c55e")

    needle_deg = max(0, min(180, value_display * 1.8))
    nx = cx + (r - 10) * math.cos(math.radians(180 - needle_deg))
    ny = cy - (r - 10) * math.sin(math.radians(180 - needle_deg))
    needle = (
        f"<line x1='{cx}' y1='{cy}' x2='{nx:.1f}' y2='{ny:.1f}' "
        f"stroke='#0f172a' stroke-width='3' stroke-linecap='round'/>"
        f"<circle cx='{cx}' cy='{cy}' r='6' fill='#0f172a'/>"
    )
    svg = f"""
    <svg viewBox='0 0 240 150' width='100%' preserveAspectRatio='xMidYMid meet'>
      {bands}
      {needle}
    </svg>
    """
    return svg, label, col


def synthesis_insights(latest: pd.Series,
                       core1: pd.DataFrame, core2: pd.DataFrame,
                       core3: pd.DataFrame, core4: pd.DataFrame,
                       team: str,
                       disc: dict, sol: float) -> dict:
    """Cross-reference DISC with pulse metrics to produce combined insights.

    Returns a dict with:
      - headline: one-sentence summary of where the team stands
      - trajectory: read on trend direction across S1->S4
      - combined_signals: list of (title, detail) cross-tab insights
      - top_priority: the single most important thing to do next
    """
    wb = latest.get("Wellbeing", np.nan)
    cog = latest.get("CogLoad", np.nan)
    cap = latest.get("Capacity", np.nan)
    collab = latest.get("Collaboration", np.nan)
    safety = latest.get("Safety", np.nan)
    capab = latest.get("Capability", np.nan)

    dom = disc.get("dominant")
    pcts = disc.get("percents", {"D": 0, "I": 0, "S": 0, "C": 0})
    div = disc.get("diversity", 0.0)

    # ---- Headline: one-sentence snapshot -----------------------------------
    # Build a qualitative health bucket from the pulse metrics.
    pulse_scores = [s for s in [wb, cap, collab, safety, capab] if pd.notna(s)]
    # CogLoad is inverted
    if pd.notna(cog):
        pulse_scores.append(100 - cog)
    pulse_mean = float(np.mean(pulse_scores)) if pulse_scores else np.nan

    if pd.isna(pulse_mean):
        pulse_bucket = "unclear"
    elif pulse_mean >= GOOD:
        pulse_bucket = "healthy"
    elif pulse_mean >= OK:
        pulse_bucket = "mixed"
    else:
        pulse_bucket = "under strain"

    if pd.isna(sol):
        sol_bucket = "unclear"
    elif sol >= GOOD:
        sol_bucket = "strong"
    elif sol >= OK:
        sol_bucket = "on track"
    else:
        sol_bucket = "lagging"

    dom_label = DISC_META[dom]["name"] if dom else "unclassified"
    # Grammar: pick 'a' vs 'an' and rephrase awkward adjectival buckets.
    pulse_phrase = {
        "healthy": "a healthy team",
        "mixed": "a team with mixed signals",
        "under strain": "a team under strain",
        "unclear": "a team with incomplete data",
    }.get(pulse_bucket, "a team")
    sol_phrase = {
        "strong": "a strong solution",
        "on track": "a solution on track",
        "lagging": "a lagging solution",
        "unclear": "an unclear solution score",
    }.get(sol_bucket, "an unclear solution")
    headline = (
        f"{pulse_phrase.capitalize()} with {sol_phrase}, "
        f"tilted toward {dom_label} style."
    )

    # ---- Trajectory: is the team trending up, down, or flat? ---------------
    # Compare the latest available wave (S4 preferred) vs S1 on wellbeing +
    # (inverted) cog load as a composite.
    def _composite(core):
        if team not in core.index:
            return np.nan
        row = core.loc[team]
        bits = []
        if "Wellbeing" in core.columns and pd.notna(row["Wellbeing"]):
            bits.append(row["Wellbeing"])
        if "CogLoad" in core.columns and pd.notna(row["CogLoad"]):
            bits.append(100 - row["CogLoad"])
        return float(np.mean(bits)) if bits else np.nan

    c1 = _composite(core1)
    # Pick the latest wave with a composite reading, so a team that missed S4
    # but answered S3 still gets a trajectory.
    c_latest = np.nan
    latest_wave_name = ""
    for core, name in ((core4, "Survey 4"), (core3, "Survey 3"),
                       (core2, "Survey 2")):
        c = _composite(core)
        if pd.notna(c):
            c_latest = c
            latest_wave_name = name
            break

    if pd.isna(c1) or pd.isna(c_latest):
        trajectory = (
            "Not enough wave-over-wave data to read the trajectory yet."
        )
    else:
        delta = c_latest - c1
        if delta >= 8:
            trajectory = (
                f"Trending up - composite wellbeing+load is {delta:+.0f} "
                f"points at {latest_wave_name} vs Survey 1. "
                "Keep doing what's working."
            )
        elif delta <= -8:
            trajectory = (
                f"Trending down - composite wellbeing+load is {delta:+.0f} "
                f"points at {latest_wave_name} vs Survey 1. "
                "Check in with the team."
            )
        else:
            trajectory = (
                f"Stable - composite wellbeing+load moved only {delta:+.0f} "
                f"points at {latest_wave_name} vs Survey 1. Steady ship."
            )

    # ---- Combined DISC x Pulse signals -------------------------------------
    signals: list[tuple[str, str]] = []

    # 1. High C + high cog load = rigour becoming overhead
    if pcts.get("C", 0) >= 35 and pd.notna(cog) and cog >= 65:
        signals.append((
            "Rigour is tipping into overhead",
            "The team is strongly Conscientious AND cognitive load is high "
            f"({cog:.0f}). Deep thinkers under high load tend to dig deeper "
            "instead of cutting scope. Explicitly permit 'good enough' "
            "decisions this week."
        ))

    # 2. High I + low solution score = ideation not converging
    if pcts.get("I", 0) >= 35 and pd.notna(sol) and sol < OK:
        signals.append((
            "Ideas are flowing but not landing",
            f"An Influence-heavy team ({pcts['I']:.0f}%) with a low solution "
            f"score ({sol:.0f}) usually means divergence outpacing "
            "convergence. Pick ONE idea to prototype by end of day."
        ))

    # 3. High D + low safety = dominance crowding out quiet voices
    if pcts.get("D", 0) >= 35 and pd.notna(safety) and safety < GOOD:
        signals.append((
            "Strong drivers, quieter voices",
            f"Dominance is {pcts['D']:.0f}% of the team but psychological "
            f"safety sits at {safety:.0f}. High-D teams can unintentionally "
            "silence detail-checkers. Round-robin the next retro."
        ))

    # 4. High S + dropping wellbeing = anchors carrying stress silently
    if pcts.get("S", 0) >= 35 and pd.notna(wb) and wb < OK:
        signals.append((
            "Anchors may be absorbing stress",
            f"Steadiness is {pcts['S']:.0f}% of the team and wellbeing has "
            f"dropped to {wb:.0f}. S-types tend to absorb strain without "
            "flagging it. Make space for one-to-ones this week."
        ))

    # 5. Low diversity (stylistic narrowness) + mixed/lagging solution
    if div < 0.5 and pd.notna(sol) and sol < GOOD:
        missing = [DISC_META[k]["name"] for k in "DISC" if pcts.get(k, 0) == 0]
        if missing:
            signals.append((
                "Stylistic blind spot",
                f"The team is stylistically narrow (diversity {div:.0%}) and "
                f"missing {' / '.join(missing)}. The solution score "
                f"({sol:.0f}) hints the missing style's strengths are exactly "
                "what would move it up. Borrow perspective from another team."
            ))

    # 6. Balanced DISC + strong pulse = let them run
    if div >= 0.8 and pulse_bucket == "healthy" and sol_bucket in ("strong", "on track"):
        signals.append((
            "This team is firing on all cylinders",
            "Diverse DISC mix, healthy pulse, solid solution score. "
            "The best thing leadership can do is stay out of the way and "
            "make sure they present in the final showcase."
        ))

    # 7. Capability low but capacity ok = skill gap not bandwidth
    if pd.notna(capab) and capab < OK and pd.notna(cap) and cap >= OK:
        signals.append((
            "Skill gap, not a bandwidth gap",
            f"Capacity is fine ({cap:.0f}) but capability reads low "
            f"({capab:.0f}). More hours won't fix this; pair the team with "
            "someone who has the missing expertise for a 30-minute consult."
        ))

    # 8. Strong collaboration - call it out so leadership notices the asset
    if pd.notna(collab) and collab >= GOOD and pd.notna(safety) and safety >= GOOD:
        signals.append((
            "Collaboration is this team's superpower",
            f"Both collaboration ({collab:.0f}) and psychological safety "
            f"({safety:.0f}) are strong. That's a rare combination and the "
            "platform for everything else this team wants to achieve."
        ))

    # For wave-over-wave signals, pick the latest available wave after S1.
    def _latest_metric(metric):
        for core in (core4, core3, core2):
            if team in core.index and metric in core.columns:
                v = core.loc[team, metric]
                if pd.notna(v):
                    return v
        return np.nan

    # 9. Wellbeing trajectory dropping (S1 -> latest wave)
    if team in core1.index and "Wellbeing" in core1.columns:
        w1 = core1.loc[team, "Wellbeing"]
        w_latest = _latest_metric("Wellbeing")
        if pd.notna(w1) and pd.notna(w_latest) and (w_latest - w1) <= -8:
            signals.append((
                "Wellbeing is slipping",
                f"Wellbeing dropped from {w1:.0f} to {w_latest:.0f} across "
                f"the hack ({w_latest - w1:+.0f}). Whatever's happening, "
                "the team is noticing it."
            ))

    # 10. Cognitive load climbing (S1 -> latest wave)
    if team in core1.index and "CogLoad" in core1.columns:
        cl1 = core1.loc[team, "CogLoad"]
        cl_latest = _latest_metric("CogLoad")
        if pd.notna(cl1) and pd.notna(cl_latest) and (cl_latest - cl1) >= 10:
            signals.append((
                "Load is piling up",
                f"Cognitive load rose from {cl1:.0f} to {cl_latest:.0f} "
                "across the hack. The team is absorbing more than it's "
                "shipping - something has to give."
            ))

    # 11. DISC dominant style that MATCHES the team's current need
    # (e.g. lagging solution + high D = drivers in the right seat)
    if dom == "D" and pd.notna(sol) and sol < OK:
        signals.append((
            "The right style for the current moment",
            f"Lagging solution score ({sol:.0f}) + Dominance-led team = the "
            "drive to pivot fast is already here. Give them permission to "
            "make a bold call."
        ))
    elif dom == "C" and pd.notna(cap) and cap >= GOOD and pd.notna(capab) and capab >= GOOD:
        signals.append((
            "Rigour meeting strong fundamentals",
            f"Conscientious-led team with capacity ({cap:.0f}) and capability "
            f"({capab:.0f}) both strong - this is the setup for a polished, "
            "well-documented deliverable."
        ))

    # 12. Both capacity AND capability low = team is simply stretched thin
    if (pd.notna(cap) and cap < OK and pd.notna(capab) and capab < OK):
        signals.append((
            "Stretched on both sides",
            f"Capacity ({cap:.0f}) and capability ({capab:.0f}) are both "
            "under pressure. This is the hardest combination - the team is "
            "under-resourced AND not confident in its expertise. Narrow the "
            "scope to something the current mix can actually ship."
        ))

    # 13. Wellbeing strongly above average = worth naming explicitly
    if pd.notna(wb) and wb >= GOOD and pulse_bucket != "under strain":
        signals.append((
            "Morale is holding up well",
            f"Wellbeing at {wb:.0f} is a real asset - engaged teams ship "
            "better work. Protect it by not letting scope creep erode "
            "the energy."
        ))

    # Fallback if nothing fired
    if not signals:
        signals.append((
            "No sharp cross-signals",
            "Nothing in the DISC profile combines dangerously with the "
            "pulse metrics right now. Keep monitoring."
        ))

    # ---- Top priority: pick the single most urgent action ------------------
    # Order of urgency: safety < OK, then wellbeing < OK, then cog > 70,
    # then solution < OK, then capacity/capability issues.
    top_priority = None
    if pd.notna(safety) and safety < OK:
        top_priority = (
            "Run a psychological safety check-in this week",
            f"Safety is at {safety:.0f} - the lowest leverage point. Nothing "
            "else on this dashboard improves until people feel safe to "
            "speak."
        )
    elif pd.notna(wb) and wb < OK:
        top_priority = (
            "Protect wellbeing before pushing on delivery",
            f"Wellbeing is at {wb:.0f}. Pushing harder now compounds the "
            "problem. Pull scope, not people."
        )
    elif pd.notna(cog) and cog > 70:
        top_priority = (
            "Cut scope - cognitive load is in the red",
            f"Cog load at {cog:.0f} means decisions get worse, not faster. "
            "Drop the two lowest-priority items on the backlog today."
        )
    elif pd.notna(sol) and sol < OK:
        top_priority = (
            "Test the solution with a real user today",
            f"Solution score {sol:.0f} suggests the team isn't confident "
            "the idea works. The fastest way to find out is to stop "
            "iterating internally and get external signal."
        )
    elif pd.notna(capab) and capab < OK:
        top_priority = (
            "Bring in targeted expertise for 30 minutes",
            f"Capability at {capab:.0f} - the team knows they're missing "
            "something. One outside consult is cheaper than a week of "
            "guessing."
        )
    else:
        top_priority = (
            "Keep the rhythm - this team is in good shape",
            "No red flags. Focus energy on sharpening the final demo "
            "rather than changing course."
        )

    return {
        "headline": headline,
        "trajectory": trajectory,
        "combined_signals": signals,
        "top_priority": top_priority,
        "pulse_mean": pulse_mean,
        "pulse_bucket": pulse_bucket,
        "sol_bucket": sol_bucket,
    }


# --------------------------------------------------------------------------- #
# Team motivational quote
# --------------------------------------------------------------------------- #
#
# Quotes are grouped by the team's archetype (derived from pulse bucket +
# solution bucket + whether they're trending up/down) and then by dominant
# DISC style where the quote benefits from style-specific framing.
#
# A deterministic hash of the team ID picks which quote in the bucket to
# use, so the same team sees the same quote on repeat runs.

_QUOTES_BY_ARCHETYPE = {
    # ---- Thriving teams: strong pulse + strong/on-track solution ----------
    "thriving": {
        "D": [
            ("The best teams don't just win - they decide who gets to play "
             "next. Keep making bold calls.",
             "Team dynamics literature"),
            ("Momentum is a choice you make every morning. You've been "
             "choosing well.",
             "James Clear (paraphrased)"),
        ],
        "I": [
            ("Great ideas deserve great execution. You've got both - make "
             "sure the world sees it.",
             "Adapted from Steve Jobs"),
            ("Energy is contagious. You're giving the rest of the cohort "
             "something to catch.",
             "Team coaching adage"),
        ],
        "S": [
            ("Trust builds slowly and compounds fast. Yours is compounding "
             "right now.",
             "Patrick Lencioni"),
            ("The quiet teams that take care of each other are the ones who "
             "ship on time, every time.",
             "Agile retrospective wisdom"),
        ],
        "C": [
            ("Precision plus follow-through is the rarest combination in "
             "any hack. You have both.",
             "Engineering folk wisdom"),
            ("Rigour isn't slow - it's the only way to go fast twice.",
             "Adapted from Robert C. Martin"),
        ],
        # Fallback used when dominant style is None or unrecognised
        "*": [
            ("The work you're doing this week is the work you'll remember "
             "for years. Finish strong.",
             "Hack coach"),
            ("You're not just building a solution - you're building the "
             "team who builds the next one.",
             "Team coaching adage"),
        ],
    },

    # ---- Stable teams: mixed pulse, on-track solution ---------------------
    "steady": {
        "D": [
            ("The goal isn't to move faster. It's to decide faster about "
             "what matters.",
             "Jeff Bezos (paraphrased)"),
            ("You don't need perfect conditions. You need one clear "
             "priority and the will to defend it.",
             "Delivery playbook"),
        ],
        "I": [
            ("Every good team has a moment where they choose depth over "
             "breadth. This is yours.",
             "Adapted from Seth Godin"),
            ("Enthusiasm is easy. Enthusiasm plus focus is what ships "
             "great work.",
             "Product management folklore"),
        ],
        "S": [
            ("The teams that keep showing up are the teams who figure it "
             "out. Keep showing up.",
             "Coaching axiom"),
            ("You don't have to be loud to be heard. You have to be "
             "consistent.",
             "Team dynamics literature"),
        ],
        "C": [
            ("Perfect is the enemy of shipped. Ship the 80% - polish it "
             "after feedback.",
             "Voltaire (paraphrased)"),
            ("Your attention to detail is an asset. Protect it by choosing "
             "which details actually matter.",
             "Design thinking folklore"),
        ],
        "*": [
            ("The middle of a hack is where teams either find their pace "
             "or lose their way. You're finding it.",
             "Hack coach"),
            ("Progress isn't always visible. Sometimes it's the absence of "
             "chaos. That counts.",
             "Retrospective wisdom"),
        ],
    },

    # ---- Struggling teams: under strain OR lagging solution ---------------
    "struggling": {
        "D": [
            ("The fastest way through a wall isn't more force - it's "
             "finding the door. Pivot boldly.",
             "Negotiation folklore"),
            ("Drive is a tool, not an identity. Use it on the right "
             "problem, and use it less on the wrong one.",
             "Leadership coaching adage"),
        ],
        "I": [
            ("Stop pitching it. Test it. The idea you love least after a "
             "real user tries it is the idea to keep.",
             "Lean startup principle"),
            ("Energy without focus is just noise. Pick one thread and "
             "follow it to the end today.",
             "Product coaching"),
        ],
        "S": [
            ("Real teams earn trust by naming what they don't know yet. "
             "Name it today - it gets easier from here.",
             "Psychological safety research"),
            ("Being steady under pressure is a superpower. Being honest "
             "about the pressure is a greater one.",
             "Resilience literature"),
        ],
        "C": [
            ("You don't need more data. You need one decision, made with "
             "what you have, shipped by end of day.",
             "Decision-science folklore"),
            ("Analysis is only valuable when it converts to action. What's "
             "the smallest action your best analysis supports?",
             "Management coaching"),
        ],
        "*": [
            ("Every great team has a day where nothing works. The ones who "
             "ship are the ones who come back tomorrow anyway.",
             "Hack coach"),
            ("You're not behind. You're gathering the evidence you need "
             "to pivot well. Act on it.",
             "Delivery coaching"),
        ],
    },

    # ---- Unknown / incomplete data ----------------------------------------
    "unclear": {
        "*": [
            ("Every team has a story the data hasn't captured yet. Keep "
             "telling yours through the work.",
             "Hack coach"),
            ("The best questions start when the dashboard runs out. "
             "What's your team talking about that this doesn't show?",
             "Team coaching adage"),
        ],
    },
}


def _pick_archetype(synth: dict, sol: float) -> str:
    """Classify the team into one of four archetypes for quote selection."""
    pulse = synth.get("pulse_bucket", "unclear")
    sol_b = synth.get("sol_bucket", "unclear")

    if pulse == "unclear":
        return "unclear"
    # Struggling dominates: under strain, or a lagging solution
    if pulse == "under strain" or sol_b == "lagging":
        return "struggling"
    # Thriving: healthy + not lagging, solution strong or on track
    if pulse == "healthy" and sol_b in ("strong", "on track"):
        return "thriving"
    # Everything else is steady
    return "steady"


def pick_team_quote(team: str, synth: dict, disc: dict,
                    sol: float) -> tuple[str, str]:
    """Return (quote_text, attribution) tailored to this team."""
    archetype = _pick_archetype(synth, sol)
    dom = disc.get("dominant") or "*"

    bucket = _QUOTES_BY_ARCHETYPE.get(archetype, {})
    candidates = bucket.get(dom) or bucket.get("*") or [
        ("Keep going - the work matters even when the scoreboard is noisy.",
         "Hack coach"),
    ]

    # Deterministic pick so the same team gets the same quote each run.
    # int(team) works for our 2-digit team IDs; hash fallback is safer.
    try:
        idx = int(team) % len(candidates)
    except (ValueError, TypeError):
        idx = hash(team) % len(candidates)
    return candidates[idx]


# --------------------------------------------------------------------------- #
# AI Summary (Ollama)
# --------------------------------------------------------------------------- #

import re as _re

# Lazy-import the ollama package so the script still runs even if the user
# hasn't installed it. _OLLAMA_IMPORT_ERROR captures any import failure so we
# can surface a friendly reason in the dashboard.
try:
    import ollama as _ollama
    _OLLAMA_IMPORT_ERROR: str | None = None
except Exception as _e:
    _ollama = None
    _OLLAMA_IMPORT_ERROR = f"ollama package not available ({_e})"


def _build_ai_prompt(team: str, latest: pd.Series, disc: dict,
                     synth: dict, sol: float) -> str:
    """Compact, structured prompt for a small local model.

    The rule-based synthesis has already done the analytical work - the
    model's job is to weave it into readable prose, not to re-derive facts.
    """
    # Format metrics with 'n/a' for missing values so the model doesn't
    # hallucinate numbers.
    def fmt(v):
        return f"{v:.0f}" if pd.notna(v) else "n/a"

    metrics = {
        "wellbeing":             fmt(latest.get("Wellbeing")),
        "team_access":           fmt(latest.get("Access")),
        "productivity":          fmt(latest.get("Productivity")),
        "capacity":              fmt(latest.get("Capacity")),
        "capability":            fmt(latest.get("Capability")),
        "collaboration":         fmt(latest.get("Collaboration")),
        "psychological_safety":  fmt(latest.get("Safety")),
        "cognitive_load":        fmt(latest.get("CogLoad")),
        "solution_score":        fmt(sol),
    }

    # DISC percentages
    pcts = disc.get("percents", {})
    disc_mix = {k: f"{pcts.get(k, 0):.0f}%" for k in "DISC"}
    dominant = disc.get("dominant") or "none"
    diversity = f"{disc.get('diversity', 0) * 100:.0f}%"

    # Rule-based findings (the LLM can borrow language from these)
    findings = "\n".join(
        f"- {title}: {detail}"
        for title, detail in synth.get("combined_signals", [])
    ) or "- (no specific cross-signals flagged)"

    priority_t, priority_d = synth.get("top_priority", ("", ""))

    return f"""You are a team performance coach writing a short report for Team {team}.

Write ONLY plain prose. No markdown, no headings, no bullet points, no lists.
Write EXACTLY four short paragraphs separated by blank lines.

Paragraph 1 (overall read): In 2-3 sentences, describe where the team stands right now. Combine the pulse health and the DISC style mix into one coherent picture.

Paragraph 2 (what is working): In 2-3 sentences, name the team's real strengths. Cite specific numbers from the data.

Paragraph 3 (what is at risk): In 2-3 sentences, name the most important risks or friction points. Explain why the DISC mix makes these risks more or less likely.

Paragraph 4 (next action): In 1-2 sentences, give the single most important thing this team should do this week.

Use a professional, supportive tone. Do NOT invent numbers not shown below. Do NOT praise excessively. Do NOT use phrases like "as an AI" or "I think". Write as if you were a senior coach briefing a delivery lead.

DATA FOR TEAM {team}:

Pulse metrics (latest survey, 0-100 scale, higher is better except cognitive load where lower is better):
- Wellbeing: {metrics['wellbeing']}
- Team access: {metrics['team_access']}
- Productivity: {metrics['productivity']}
- Capacity: {metrics['capacity']}
- Capability: {metrics['capability']}
- Collaboration: {metrics['collaboration']}
- Psychological safety: {metrics['psychological_safety']}
- Cognitive load: {metrics['cognitive_load']} (lower is better)
- Solution score: {metrics['solution_score']}

DISC style mix (based on Personal Style x Preferred Culture, Survey 1):
- Dominance (drivers): {disc_mix['D']}
- Influence (energisers): {disc_mix['I']}
- Steadiness (anchors): {disc_mix['S']}
- Conscientious (analysts): {disc_mix['C']}
- Dominant style: {dominant}
- Style diversity: {diversity} (100% = perfectly mixed, 0% = single style)

Trajectory: {synth.get('trajectory', 'n/a')}

Rule-based cross-signals already flagged:
{findings}

Suggested top priority (you may reword this): {priority_t} - {priority_d}

Now write the four paragraphs."""


def _clean_llm_output(text: str) -> str:
    """Strip reasoning tags, markdown artefacts, and obvious junk."""
    # Drop <think>...</think> blocks if a reasoning model is ever swapped in
    text = _re.sub(r"<think>.*?</think>", "", text, flags=_re.DOTALL)
    # Remove stray markdown headers and list markers
    text = _re.sub(r"^\s*#{1,6}\s+", "", text, flags=_re.MULTILINE)
    text = _re.sub(r"^\s*[-*]\s+", "", text, flags=_re.MULTILINE)
    # Strip "Paragraph N:" labels if the model echoed them back
    text = _re.sub(r"^\s*\**paragraph\s*\d+[:.\)]?\s*\**\s*",
                   "", text, flags=_re.IGNORECASE | _re.MULTILINE)
    # Collapse >2 consecutive blank lines
    text = _re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def generate_ai_summary(team: str, latest: pd.Series, disc: dict,
                        synth: dict, sol: float) -> tuple[str, str]:
    """Call local Ollama via the `ollama` package. Returns (html, status).

    `status` is "ok" | "disabled" | "error:<reason>" so the caller can render
    a small indicator when something goes wrong.
    """
    if not AI_SUMMARY_ENABLED:
        return ("", "disabled")

    if _ollama is None:
        return ("", f"error:{_OLLAMA_IMPORT_ERROR}")

    prompt = _build_ai_prompt(team, latest, disc, synth, sol)

    try:
        response = _ollama.generate(
            model=OLLAMA_MODEL,
            prompt=prompt,
            stream=False,
            options={
                "temperature": 0.4,    # low so a 1B model stays grounded
                "top_p": 0.9,
                "num_predict": 450,    # ~4 short paragraphs
            },
        )
    except Exception as e:
        # The ollama package raises a few different exception types
        # (ConnectionError, ResponseError, etc.) - catch them all and
        # surface a short reason rather than crashing the whole run.
        reason = str(e) or type(e).__name__
        # Trim stupidly long messages
        if len(reason) > 200:
            reason = reason[:200] + "..."
        return ("", f"error:{reason}")

    # The ollama package returns a dict-like object; 'response' holds the text
    raw = (response.get("response") if isinstance(response, dict)
           else getattr(response, "response", "")) or ""
    raw = raw.strip()
    if not raw:
        return ("", "error:empty response from model")

    cleaned = _clean_llm_output(raw)
    # Break into paragraphs; skip empty ones
    paragraphs = [p.strip() for p in cleaned.split("\n\n") if p.strip()]
    if not paragraphs:
        return ("", "error:no usable paragraphs after cleaning")

    # Escape HTML-special chars in each paragraph so anything the model
    # accidentally emits (< > &) can't break the page.
    def esc(s: str) -> str:
        return (s.replace("&", "&amp;")
                 .replace("<", "&lt;")
                 .replace(">", "&gt;"))

    html_paragraphs = "".join(f"<p>{esc(p)}</p>" for p in paragraphs)
    return (html_paragraphs, "ok")



def disc_quadrant_svg(percents: dict, dominant: str | None) -> str:
    """2x2 DISC grid with bubble sizes = share of the team."""
    W, H = 360, 220
    # Grid corners - each cell is a quadrant matching the mapping:
    #   top-left=C, top-right=D, bottom-left=S, bottom-right=I
    # (task vs people on the vertical axis, pace on the horizontal)
    layout = {
        "C": (W * 0.27, H * 0.32),
        "D": (W * 0.73, H * 0.32),
        "S": (W * 0.27, H * 0.72),
        "I": (W * 0.73, H * 0.72),
    }

    parts: list[str] = []
    # Axis lines
    parts.append(
        f"<line x1='{W/2}' y1='20' x2='{W/2}' y2='{H-30}' "
        f"stroke='#cbd5e1' stroke-dasharray='4 4'/>"
        f"<line x1='20' y1='{H/2-5}' x2='{W-20}' y2='{H/2-5}' "
        f"stroke='#cbd5e1' stroke-dasharray='4 4'/>"
    )
    # Axis labels
    parts.append(
        f"<text x='{W/2}' y='14' text-anchor='middle' font-size='10' "
        f"fill='#64748b'>Task-focused</text>"
        f"<text x='{W/2}' y='{H-16}' text-anchor='middle' font-size='10' "
        f"fill='#64748b'>People-focused</text>"
        f"<text x='10' y='{H/2-8}' font-size='10' fill='#64748b' "
        f"transform='rotate(-90 10 {H/2-8})'>Structured</text>"
        f"<text x='{W-10}' y='{H/2-8}' text-anchor='end' font-size='10' "
        f"fill='#64748b' transform='rotate(-90 {W-10} {H/2-8})'>Fast-paced</text>"
    )

    # Bubbles - radius scales with share (min 14 so empty quadrants still show)
    for letter, (cx, cy) in layout.items():
        p = percents.get(letter, 0)
        r = 14 + (p / 100) * 36          # 14-50px radius
        colour = DISC_META[letter]["colour"]
        name = DISC_META[letter]["name"]
        is_dom = (letter == dominant)
        opacity = 0.9 if p > 0 else 0.25
        stroke = "#0f172a" if is_dom else "none"
        stroke_w = "2" if is_dom else "0"
        parts.append(
            f"<circle cx='{cx:.0f}' cy='{cy:.0f}' r='{r:.0f}' "
            f"fill='{colour}' opacity='{opacity}' "
            f"stroke='{stroke}' stroke-width='{stroke_w}'/>"
            f"<text x='{cx:.0f}' y='{cy:.0f}' text-anchor='middle' "
            f"dominant-baseline='central' font-size='16' font-weight='700' "
            f"fill='#fff'>{letter}</text>"
            f"<text x='{cx:.0f}' y='{cy + r + 14:.0f}' text-anchor='middle' "
            f"font-size='10' fill='#334155' font-weight='600'>"
            f"{name} - {p:.0f}%</text>"
        )

    return (
        f"<svg viewBox='0 0 {W} {H}' width='100%' "
        f"preserveAspectRatio='xMidYMid meet'>{''.join(parts)}</svg>"
    )


# --------------------------------------------------------------------------- #
# HTML template - matches the 'Team Pulse & Performance' mock-up
# --------------------------------------------------------------------------- #

HTML = r"""<!doctype html>
<html lang='en'><head><meta charset='utf-8'>
<title>Team {team} - Team Pulse &amp; Performance</title>
<style>
  * {{ box-sizing: border-box; }}
  body {{
    margin: 0; padding: 20px;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto,
                 Helvetica, Arial, sans-serif;
    background: #e5e7eb; color: #0f172a;
  }}
  .frame {{
    max-width: 980px; margin: 0 auto;
    background: #f1f5f9; border-radius: 12px;
    overflow: hidden; box-shadow: 0 6px 20px rgba(0,0,0,.1);
  }}
  .topbar {{
    background: #3b5cad; color: #fff;
    padding: 14px 22px;
    display: flex; align-items: center; justify-content: space-between;
  }}
  .topbar h1 {{ margin: 0; font-size: 18px; font-weight: 600; }}
  .dots span {{
    display: inline-block; width: 12px; height: 12px;
    border-radius: 50%; margin-left: 6px;
  }}
  .dots .g {{ background: #22c55e; }}
  .dots .y {{ background: #f59e0b; }}
  .dots .b {{ background: #93c5fd; }}

  /* ---- Tabs (pure CSS, no JS) ---------------------------------------- */
  .tabs {{ display: flex; background: #e2e8f0; border-bottom: 1px solid #cbd5e1; }}
  .tab-radio {{ display: none; }}
  .tab-label {{
    flex: 1; padding: 12px 16px; text-align: center; cursor: pointer;
    font-size: 13px; font-weight: 600; color: #475569;
    border-right: 1px solid #cbd5e1;
    transition: background .15s, color .15s;
    user-select: none;
  }}
  .tab-label:last-of-type {{ border-right: none; }}
  .tab-label:hover {{ background: #f1f5f9; color: #0f172a; }}
  .tab-panel {{ display: none; }}
  #tab1:checked ~ .tab-content #panel1,
  #tab2:checked ~ .tab-content #panel2,
  #tab3:checked ~ .tab-content #panel3 {{ display: block; }}
  #tab1:checked ~ .tabs label[for='tab1'],
  #tab2:checked ~ .tabs label[for='tab2'],
  #tab3:checked ~ .tabs label[for='tab3'] {{
    background: #f1f5f9; color: #3b5cad;
    box-shadow: inset 0 -3px 0 #3b5cad;
  }}

  /* ---- Shared panel chrome ------------------------------------------- */
  .panel {{
    background: #fff; border: 1px solid #e2e8f0;
    border-radius: 8px; overflow: hidden;
  }}
  .panel h2 {{
    margin: 0; background: #3b5cad; color: #fff;
    font-size: 13px; font-weight: 600; padding: 8px 12px;
    text-align: center;
  }}
  .panel-body {{ padding: 12px 14px; }}

  /* ---- Tab 1: Team Pulse ---------------------------------------------- */
  .kpis {{
    display: grid; grid-template-columns: repeat(3, 1fr);
    gap: 14px; padding: 18px;
  }}
  .kpi {{
    background: #fff; border: 1px solid #e2e8f0;
    border-radius: 8px; padding: 12px 16px;
  }}
  .kpi .label {{
    text-align: center; font-size: 13px; color: #64748b;
    border-bottom: 1px solid #e2e8f0; padding-bottom: 6px;
    margin-bottom: 10px; font-weight: 500;
  }}
  .kpi .body {{
    display: flex; align-items: center; justify-content: center; gap: 12px;
    min-height: 56px;
  }}
  .kpi .val {{ font-size: 26px; font-weight: 700; }}
  .kpi .trend {{ font-size: 18px; margin-left: 4px; }}
  .kpi .pct-chip {{
    display: block;
    font-size: 11px; font-weight: 600;
    margin-top: 6px; color: #94a3b8;
    text-align: center;
  }}
  .kpi .pct-chip .pct-label {{ color: #94a3b8; font-weight: 500; }}

  /* Motivational quote banner (top of Tab 1) */
  .quote-banner {{
    margin: 18px 18px 0;
    padding: 14px 20px 14px 48px;
    background: linear-gradient(90deg, #fef3c7 0%, #fffbeb 100%);
    border: 1px solid #fde68a;
    border-radius: 8px;
    position: relative;
  }}
  .quote-banner::before {{
    content: '\201C';
    position: absolute; left: 14px; top: 2px;
    font-family: Georgia, serif; font-size: 48px;
    color: #d97706; line-height: 1; opacity: 0.7;
  }}
  .quote-banner .quote-text {{
    font-family: Georgia, serif; font-style: italic;
    font-size: 14px; color: #78350f; line-height: 1.5;
    margin: 0 0 4px;
  }}
  .quote-banner .quote-attrib {{
    font-size: 11px; color: #a16207; text-align: right;
    font-style: normal;
  }}
  .quote-banner .quote-attrib::before {{ content: '\2014 '; }}
  .mid {{
    display: grid; grid-template-columns: 1fr 1.1fr 1fr;
    gap: 14px; padding: 0 18px;
  }}
  .sg-block {{ margin-bottom: 12px; }}
  .sg-block h3 {{
    margin: 0 0 8px; font-size: 13px; color: #334155;
    border-bottom: 1px solid #e2e8f0; padding-bottom: 4px;
  }}
  .sg-list {{ list-style: none; padding: 0; margin: 0; font-size: 13px; }}
  .sg-list li {{ display: flex; align-items: center; gap: 8px; padding: 3px 0; }}
  .sg-list .icon {{
    display: inline-flex; width: 18px; height: 18px; border-radius: 50%;
    align-items: center; justify-content: center; font-size: 11px;
    color: #fff; flex-shrink: 0; font-weight: 700;
  }}
  .icon-tick {{ background: #22c55e; }}
  .icon-warn {{ background: #ef4444; }}
  .chart-label {{
    font-size: 13px; color: #334155; margin: 4px 0 2px 6px; font-weight: 500;
  }}
  .gauge-q {{
    font-size: 13px; color: #334155; text-align: center; margin: 0 0 8px;
  }}
  .gauge-lbl {{
    color: #fff; font-weight: 600; font-size: 14px;
    padding: 6px 14px; border-radius: 6px; display: inline-block;
    margin-top: 4px;
  }}
  .gauge-wrap {{ text-align: center; }}
  .action-title {{
    text-align: center; font-family: Georgia, serif; font-style: italic;
    font-size: 20px; color: #334155; margin: 20px 0 10px;
  }}
  .actions {{
    display: grid; grid-template-columns: repeat(3, 1fr);
    gap: 14px; padding: 0 18px 20px;
  }}
  .action-card {{
    background: #fff; border-radius: 8px;
    border: 1px solid #e2e8f0; overflow: hidden;
  }}
  .action-card h3 {{
    margin: 0; color: #fff; padding: 8px 12px; font-size: 13px;
    font-weight: 600; text-align: center;
  }}
  .a-teamwork h3 {{ background: #22c55e; }}
  .a-load     h3 {{ background: #f59e0b; }}
  .a-solution h3 {{ background: #ef4444; }}
  .action-list {{ list-style: none; padding: 10px 14px; margin: 0; font-size: 12.5px; }}
  .action-list li {{
    padding: 6px 10px; margin-bottom: 6px;
    background: #f8fafc; border: 1px solid #e2e8f0; border-radius: 6px;
    display: flex; align-items: center; gap: 8px;
  }}
  .action-list li::before {{
    content: ''; display: inline-block; width: 12px; height: 12px;
    border: 1.5px solid currentColor; border-radius: 2px; flex-shrink: 0;
  }}
  .a-teamwork .action-list li {{ color: #16a34a; }}
  .a-load     .action-list li {{ color: #d97706; }}
  .a-solution .action-list li {{ color: #dc2626; }}
  .action-list li span {{ color: #0f172a; font-weight: 400; }}

  /* ---- Tab 2: DISC full-width ----------------------------------------- */
  .disc-hero {{ padding: 18px; }}
  .disc-hero-grid {{
    display: grid; grid-template-columns: 1.1fr 1fr; gap: 20px;
    padding: 18px; background: #fff; border: 1px solid #e2e8f0;
    border-radius: 8px;
  }}
  .disc-summary {{ font-size: 13px; color: #0f172a; line-height: 1.55; }}
  .disc-summary .dom-badge {{
    display: inline-block; padding: 4px 12px; border-radius: 999px;
    color: #fff; font-weight: 700; font-size: 13px; margin-bottom: 10px;
  }}
  .disc-meta {{
    font-size: 11px; color: #64748b; margin: 10px 0;
    display: flex; gap: 14px; flex-wrap: wrap;
  }}
  .disc-meta b {{ color: #0f172a; }}
  .disc-caveat {{
    font-size: 11px; color: #94a3b8; font-style: italic;
    padding: 10px 18px 0; text-align: center;
  }}
  .disc-cards {{
    display: grid; grid-template-columns: repeat(4, 1fr);
    gap: 12px; padding: 18px;
  }}
  .disc-card {{
    background: #fff; border: 1px solid #e2e8f0; border-radius: 8px;
    padding: 12px; position: relative;
  }}
  .disc-card.is-dom {{ border-color: #0f172a; border-width: 2px; }}
  .disc-card .letter {{
    display: inline-block; width: 32px; height: 32px; border-radius: 50%;
    color: #fff; font-weight: 700; font-size: 16px;
    text-align: center; line-height: 32px; margin-bottom: 8px;
  }}
  .disc-card .name {{
    font-size: 13px; font-weight: 600; color: #0f172a; margin-bottom: 2px;
  }}
  .disc-card .pct {{ font-size: 20px; font-weight: 700; color: #0f172a; }}
  .disc-card .blurb {{
    font-size: 11px; color: #64748b; margin-top: 6px; line-height: 1.4;
  }}

  /* ---- Tab 3: Insights / synthesis ----------------------------------- */
  .insight-wrap {{ padding: 18px; }}

  /* AI-generated narrative card (sits at top of Tab 3) */
  .ai-card {{
    background: #fff;
    border: 1px solid #e2e8f0;
    border-left: 4px solid #8b5cf6;
    border-radius: 8px;
    padding: 16px 20px 14px;
    margin-bottom: 16px;
  }}
  .ai-card .ai-header {{
    display: flex; align-items: center; gap: 8px; margin-bottom: 10px;
    font-size: 11px; font-weight: 700; color: #7c3aed;
    text-transform: uppercase; letter-spacing: 1px;
  }}
  .ai-card .ai-header::before {{
    content: ''; display: inline-block; width: 8px; height: 8px;
    border-radius: 50%; background: #8b5cf6;
    box-shadow: 0 0 0 3px rgba(139, 92, 246, 0.18);
  }}
  .ai-card p {{
    margin: 0 0 10px; font-size: 13.5px; line-height: 1.55; color: #0f172a;
  }}
  .ai-card p:last-child {{ margin-bottom: 0; }}
  .ai-card .ai-footer {{
    font-size: 10px; color: #94a3b8; font-style: italic;
    margin-top: 10px; padding-top: 10px;
    border-top: 1px dashed #e2e8f0;
  }}
  .ai-card.ai-error {{ border-left-color: #cbd5e1; }}
  .ai-card.ai-error .ai-header {{ color: #64748b; }}
  .ai-card.ai-error .ai-header::before {{
    background: #cbd5e1; box-shadow: 0 0 0 3px rgba(203, 213, 225, 0.4);
  }}
  .headline-card {{
    background: linear-gradient(135deg, #3b5cad 0%, #1e3a8a 100%);
    color: #fff; border-radius: 8px; padding: 18px 22px; margin-bottom: 16px;
  }}
  .headline-card .eyebrow {{
    font-size: 11px; text-transform: uppercase; letter-spacing: 1px;
    opacity: 0.8; margin-bottom: 6px;
  }}
  .headline-card .headline {{ font-size: 18px; font-weight: 600; line-height: 1.35; }}
  .headline-card .trajectory {{
    font-size: 13px; opacity: 0.92; margin-top: 10px;
    padding-top: 10px; border-top: 1px solid rgba(255,255,255,0.25);
  }}

  .priority-card {{
    background: #fff; border-left: 4px solid #ef4444;
    border: 1px solid #e2e8f0; border-radius: 8px;
    padding: 14px 18px; margin-bottom: 16px;
  }}
  .priority-card .eyebrow {{
    font-size: 11px; text-transform: uppercase; letter-spacing: 1px;
    color: #dc2626; font-weight: 700; margin-bottom: 4px;
  }}
  .priority-card h3 {{
    margin: 0 0 6px; font-size: 16px; color: #0f172a; font-weight: 600;
  }}
  .priority-card p {{
    margin: 0; font-size: 13px; color: #334155; line-height: 1.5;
  }}

  .signals-grid {{
    display: grid; grid-template-columns: 1fr 1fr; gap: 12px;
    margin-bottom: 16px;
  }}
  .signal-card {{
    background: #fff; border: 1px solid #e2e8f0; border-radius: 8px;
    padding: 14px 16px;
  }}
  .signal-card h4 {{
    margin: 0 0 6px; font-size: 13px; color: #3b5cad; font-weight: 600;
    display: flex; align-items: center; gap: 6px;
  }}
  .signal-card h4::before {{
    content: ''; display: inline-block; width: 6px; height: 6px;
    background: #3b5cad; border-radius: 50%;
  }}
  .signal-card p {{
    margin: 0; font-size: 12.5px; color: #334155; line-height: 1.5;
  }}

  .mini-stats {{
    display: grid; grid-template-columns: repeat(4, 1fr);
    gap: 10px; margin-bottom: 16px;
  }}
  .mini-stat {{
    background: #fff; border: 1px solid #e2e8f0; border-radius: 8px;
    padding: 10px 12px; text-align: center;
  }}
  .mini-stat .label {{
    font-size: 10px; color: #64748b; text-transform: uppercase;
    letter-spacing: 0.5px; margin-bottom: 4px;
  }}
  .mini-stat .val {{
    font-size: 18px; font-weight: 700; color: #0f172a;
  }}
  .mini-stat .val.good {{ color: #16a34a; }}
  .mini-stat .val.warn {{ color: #d97706; }}
  .mini-stat .val.bad  {{ color: #dc2626; }}

  .footer {{
    padding: 8px 18px 14px; color: #64748b; font-size: 11px;
    text-align: center;
  }}
</style></head>
<body>
<div class='frame'>

  <div class='topbar'>
    <h1>Team {team} - Team Pulse &amp; Performance</h1>
    <div class='dots'>
      <span class='g'></span><span class='y'></span><span class='b'></span>
    </div>
  </div>

  <!-- Radio inputs control which tab is shown -->
  <input type='radio' name='tabs' id='tab1' class='tab-radio' checked>
  <input type='radio' name='tabs' id='tab2' class='tab-radio'>
  <input type='radio' name='tabs' id='tab3' class='tab-radio'>

  <div class='tabs'>
    <label class='tab-label' for='tab1'>Team Pulse</label>
    <label class='tab-label' for='tab2'>DISC Profile</label>
    <label class='tab-label' for='tab3'>Insights &amp; Synthesis</label>
  </div>

  <div class='tab-content'>

    <!-- ==================== TAB 1: TEAM PULSE =================== -->
    <div class='tab-panel' id='panel1'>
      <div class='quote-banner'>
        <p class='quote-text'>{quote_text}</p>
        <div class='quote-attrib'>{quote_attrib}</div>
      </div>
      <div class='kpis'>
        <div class='kpi'>
          <div class='label'>Wellbeing</div>
          <div class='body'>{wb_face}<span class='val'>{wb_label}</span></div>
          <span class='pct-chip'>{wb_pct_chip}</span>
        </div>
        <div class='kpi'>
          <div class='label'>Team Access</div>
          <div class='body'>{acc_face}<span class='val'>{acc_val}</span>
            <span class='trend' style='color:{acc_col}'>{acc_arrow}</span>
          </div>
          <span class='pct-chip'>{acc_pct_chip}</span>
        </div>
        <div class='kpi'>
          <div class='label'>Cognitive Load</div>
          <div class='body'>{load_face}<span class='val'>{load_val}</span>
            <span class='trend' style='color:{load_col}'>{load_arrow}</span>
          </div>
          <span class='pct-chip'>{load_pct_chip}</span>
        </div>
      </div>

      <div class='mid'>
        <div class='panel'>
          <h2>Team Strengths &amp; Gaps</h2>
          <div class='panel-body'>
            <div class='sg-block'>
              <h3>We're Great At:</h3>
              <ul class='sg-list'>{strengths_html}</ul>
            </div>
            <div class='sg-block'>
              <h3>We Can Improve:</h3>
              <ul class='sg-list'>{gaps_html}</ul>
            </div>
          </div>
        </div>

        <div class='panel'>
          <h2>Team Mood Tracker</h2>
          <div class='panel-body'>
            <div class='chart-label'>How We're Feeling (Wellbeing - higher is better)</div>
            {mood_chart}
            <div class='chart-label'>Our Stress Level (Cognitive Load - lower is better)</div>
            {stress_chart}
          </div>
        </div>

        <div class='panel'>
          <h2>How Strong Is Our Solution?</h2>
          <div class='panel-body gauge-wrap'>
            <p class='gauge-q'>Solution fit, innovation &amp; ambition</p>
            {gauge}
            <div class='gauge-lbl' style='background:{gauge_col}'>{gauge_lbl}</div>
          </div>
        </div>
      </div>

      <div class='action-title'>Take Action!</div>
      <div class='actions'>
        <div class='action-card a-teamwork'>
          <h3>Strengthen Teamwork</h3>
          <ul class='action-list'>{teamwork_html}</ul>
        </div>
        <div class='action-card a-load'>
          <h3>Ease the Load</h3>
          <ul class='action-list'>{load_html}</ul>
        </div>
        <div class='action-card a-solution'>
          <h3>Boost Our Solution</h3>
          <ul class='action-list'>{solution_html}</ul>
        </div>
      </div>
    </div>

    <!-- ==================== TAB 2: DISC PROFILE =================== -->
    <div class='tab-panel' id='panel2'>
      <div class='disc-hero'>
        <div class='disc-hero-grid'>
          <div>{disc_quadrant}</div>
          <div class='disc-summary'>
            <span class='dom-badge' style='background:{disc_dom_colour}'>
              {disc_dom_letter} - {disc_dom_name}
            </span>
            <div>{disc_commentary}</div>
            <div class='disc-meta'>
              <span><b>{disc_n}</b> profiled</span>
              <span>Diversity: <b>{disc_diversity_pct}%</b></span>
            </div>
          </div>
        </div>
      </div>

      <div class='disc-cards'>
        {disc_cards_html}
      </div>

      <div class='disc-caveat'>
        Two-axis approximation from Survey 1 (Personal Style x Preferred
        Culture) - not a full DISC assessment. Personal Style &lt; 50 = task-
        focused, &gt;= 50 = people-focused. Preferred Culture &lt; 50 =
        structured, &gt;= 50 = fast-paced.
      </div>
    </div>

    <!-- ==================== TAB 3: INSIGHTS =================== -->
    <div class='tab-panel' id='panel3'>
      <div class='insight-wrap'>

        {ai_summary_block}

        <div class='headline-card'>
          <div class='eyebrow'>Where this team stands</div>
          <div class='headline'>{synth_headline}</div>
          <div class='trajectory'>{synth_trajectory}</div>
        </div>

        <div class='mini-stats'>
          <div class='mini-stat'>
            <div class='label'>Pulse (composite)</div>
            <div class='val {pulse_cls}'>{pulse_val}</div>
          </div>
          <div class='mini-stat'>
            <div class='label'>Solution score</div>
            <div class='val {sol_cls}'>{sol_val}</div>
          </div>
          <div class='mini-stat'>
            <div class='label'>Dominant style</div>
            <div class='val' style='color:{disc_dom_colour}'>{disc_dom_letter}</div>
          </div>
          <div class='mini-stat'>
            <div class='label'>DISC diversity</div>
            <div class='val'>{disc_diversity_pct}%</div>
          </div>
        </div>

        <div class='priority-card'>
          <div class='eyebrow'>Top priority this week</div>
          <h3>{priority_title}</h3>
          <p>{priority_detail}</p>
        </div>

        <div class='signals-grid'>
          {signals_html}
        </div>
      </div>
    </div>

  </div>

  <div class='footer'>
    Team {team} - {n} respondents across Surveys 1-4.
    Thresholds: green &gt;= {good}, amber &gt;= {ok}, red &lt; {ok}. Cognitive
    Load is inverted (low is good). Trend arrows compare the latest survey
    vs the one before it.
  </div>
</div>
</body></html>
"""
HTML = HTML.replace("{good}", str(GOOD)).replace("{ok}", str(OK))


# --------------------------------------------------------------------------- #
# Per-team render
# --------------------------------------------------------------------------- #

def _latest_row(team: str,
                core1: pd.DataFrame,
                core2: pd.DataFrame,
                core3: pd.DataFrame,
                core4: pd.DataFrame) -> pd.Series:
    """Latest reading per team: S4 first, cascading back through S3, S2, S1."""
    s4 = core4.loc[team] if team in core4.index else pd.Series(dtype=float)
    s3 = core3.loc[team] if team in core3.index else pd.Series(dtype=float)
    s2 = core2.loc[team] if team in core2.index else pd.Series(dtype=float)
    s1 = core1.loc[team] if team in core1.index else pd.Series(dtype=float)
    return s4.combine_first(s3).combine_first(s2).combine_first(s1)


def build_dashboard(team: str,
                    s1: pd.DataFrame,
                    s2: pd.DataFrame,
                    s3: pd.DataFrame,
                    s4: pd.DataFrame,
                    core1: pd.DataFrame,
                    core2: pd.DataFrame,
                    core3: pd.DataFrame,
                    core4: pd.DataFrame,
                    challenge2: pd.DataFrame,
                    challenge3: pd.DataFrame,
                    challenge4: pd.DataFrame,
                    ind1: pd.DataFrame,
                    ind2: pd.DataFrame,
                    ind3: pd.DataFrame,
                    ind4: pd.DataFrame) -> str:
    """Produce one team's HTML dashboard. All params are pre-computed once."""

    # --- Latest (S4) reading with S3 / S2 / S1 fallback ---------------------
    latest = _latest_row(team, core1, core2, core3, core4)
    # Prior for trend arrows = the most recent available prior wave.
    # Prefer S3 (the wave just before S4); fall back to S2 then S1.
    if team in core3.index:
        prior = core3.loc[team]
    elif team in core2.index:
        prior = core2.loc[team]
    elif team in core1.index:
        prior = core1.loc[team]
    else:
        prior = pd.Series(dtype=float)

    # --- Top KPI cards ------------------------------------------------------
    wb = latest.get("Wellbeing", np.nan)
    wb_col, wb_label, wb_face = mood_label(wb)
    wb_prior = prior.get("Wellbeing", np.nan)
    wb_pct_text, wb_pct_col = _pct_change(wb, wb_prior)

    access = latest.get("Access", np.nan)
    acc_col, _, acc_face = access_label(access)
    acc_prior = prior.get("Access", np.nan)
    acc_arrow = _trend_arrow(access, acc_prior)
    acc_pct_text, acc_pct_col = _pct_change(access, acc_prior)

    cog = latest.get("CogLoad", np.nan)
    load_col, _, load_face = load_label(cog)
    cog_prior = prior.get("CogLoad", np.nan)
    load_arrow = _trend_arrow(cog, cog_prior, higher_is_worse=True)
    load_pct_text, load_pct_col = _pct_change(cog, cog_prior,
                                              higher_is_worse=True)

    # Build the KPI percent-change chips. Empty string hides the chip
    # entirely when there's no prior reading to compare against.
    def _chip(pct_text: str, pct_col: str) -> str:
        if not pct_text:
            return ""
        return (
            f"<span style='color:{pct_col}'>{pct_text}</span>"
            f" <span class='pct-label'>vs last survey</span>"
        )

    wb_pct_chip = _chip(wb_pct_text, wb_pct_col)
    acc_pct_chip = _chip(acc_pct_text, acc_pct_col)
    load_pct_chip = _chip(load_pct_text, load_pct_col)

    # --- Strengths & gaps ---------------------------------------------------
    strengths, gaps = strengths_and_gaps(latest)
    strengths_html = "".join(
        f"<li><span class='icon icon-tick'>\u2713</span>{s}</li>"
        for s in strengths
    ) or "<li><span class='icon icon-tick'>\u2713</span>Balanced performance</li>"
    gaps_html = "".join(
        f"<li><span class='icon icon-warn'>!</span>{g}</li>"
        for g in gaps
    ) or "<li><span class='icon icon-tick'>\u2713</span>No major gaps</li>"

    # --- Mood & stress tracker (bar chart: one bar per survey) --------------
    # Four team-mean values per metric, one per survey wave.
    def _wave_value(core, metric):
        if team in core.index and metric in core.columns:
            return core.loc[team, metric]
        return np.nan

    wb_bars = [_wave_value(c, "Wellbeing")
               for c in (core1, core2, core3, core4)]
    cl_bars = [_wave_value(c, "CogLoad")
               for c in (core1, core2, core3, core4)]

    mood_chart = bar_chart_svg(
        wb_bars, stroke="#16a34a", fill="#86efac",
        labels=("Survey 1", "Survey 2", "Survey 3", "Survey 4"),
    )
    stress_chart = bar_chart_svg(
        cl_bars, stroke="#ea580c", fill="#fdba74",
        labels=("Survey 1", "Survey 2", "Survey 3", "Survey 4"),
        inverted=True,
    )

    # --- Solution gauge -----------------------------------------------------
    # Use the most recent available challenge answers: S4 -> S3 -> S2.
    if team in challenge4.index:
        sol_row = challenge4.loc[team]
    elif team in challenge3.index:
        sol_row = challenge3.loc[team]
    elif team in challenge2.index:
        sol_row = challenge2.loc[team]
    else:
        sol_row = pd.Series(dtype=float)
    sol = solution_score(sol_row)
    gauge, gauge_lbl, gauge_col = gauge_svg(sol)

    # --- Action buckets -----------------------------------------------------
    buckets = actions_for(latest, sol)
    teamwork_html = "".join(
        f"<li><span>{a}</span></li>" for a in buckets["Strengthen Teamwork"]
    )
    load_html = "".join(
        f"<li><span>{a}</span></li>" for a in buckets["Ease the Load"]
    )
    solution_html = "".join(
        f"<li><span>{a}</span></li>" for a in buckets["Boost Our Solution"]
    )

    n_respondents = (
        s1[s1["Team"] == team]["Individual"].nunique()
        + s2[s2["Team"] == team]["Individual"].nunique()
        + s3[s3["Team"] == team]["Individual"].nunique()
        + s4[s4["Team"] == team]["Individual"].nunique()
    )

    # --- DISC team-dynamics profile (from Survey 1) -------------------------
    disc = team_disc_profile(s1, team)
    disc_quadrant = disc_quadrant_svg(disc["percents"], disc["dominant"])
    if disc["dominant"]:
        disc_dom_letter = disc["dominant"]
        disc_dom_name = DISC_META[disc["dominant"]]["name"]
        disc_dom_colour = DISC_META[disc["dominant"]]["colour"]
    else:
        disc_dom_letter = "?"
        disc_dom_name = "Unknown"
        disc_dom_colour = "#94a3b8"

    # Tab 2: four per-style breakdown cards
    disc_cards_html = "".join(
        f"<div class='disc-card{' is-dom' if k == disc['dominant'] else ''}'>"
        f"<span class='letter' style='background:{DISC_META[k]['colour']}'>"
        f"{k}</span>"
        f"<div class='name'>{DISC_META[k]['name']}</div>"
        f"<div class='pct'>{disc['percents'].get(k, 0):.0f}%</div>"
        f"<div class='blurb'>{DISC_META[k]['blurb']}</div>"
        f"</div>"
        for k in "DISC"
    )
    disc_diversity_pct = f"{disc['diversity'] * 100:.0f}"

    # --- Tab 3: synthesis / insights ----------------------------------------
    synth = synthesis_insights(latest, core1, core2, core3, core4,
                               team, disc, sol)

    # --- Tab 1: motivational quote -----------------------------------------
    # Placed up here because it depends on synth + disc + sol, all of which
    # are now available. Safe HTML-escape the strings since they're
    # interpolated directly into the template.
    quote_text_raw, quote_attrib_raw = pick_team_quote(team, synth, disc, sol)
    def _esc_txt(s: str) -> str:
        return (s.replace("&", "&amp;")
                 .replace("<", "&lt;")
                 .replace(">", "&gt;")
                 .replace('"', "&quot;"))
    quote_text = _esc_txt(quote_text_raw)
    quote_attrib = _esc_txt(quote_attrib_raw)

    # --- AI narrative summary (calls local Ollama) --------------------------
    ai_html, ai_status = generate_ai_summary(team, latest, disc, synth, sol)
    if ai_status == "ok":
        ai_summary_block = (
            "<div class='ai-card'>"
            "<div class='ai-header'>AI Summary</div>"
            f"{ai_html}"
            f"<div class='ai-footer'>Generated by {OLLAMA_MODEL} from "
            "the pulse and DISC data for this team. Verify against the "
            "source data before acting on specifics.</div>"
            "</div>"
        )
    elif ai_status == "disabled":
        ai_summary_block = ""   # don't render anything when turned off
    else:
        # Error path - show a small muted card so the user knows why the
        # narrative is missing, but the rest of the dashboard still works.
        ai_summary_block = (
            "<div class='ai-card ai-error'>"
            "<div class='ai-header'>AI Summary unavailable</div>"
            f"<p>The rule-based synthesis below is still available. "
            f"Reason: {ai_status[6:] if ai_status.startswith('error:') else ai_status}.</p>"
            "</div>"
        )

    def _score_cls(v: float, inverted: bool = False) -> str:
        """Map a 0-100 score to a CSS class for colour."""
        if pd.isna(v):
            return ""
        if v >= GOOD:
            return "good"
        if v >= OK:
            return "warn"
        return "bad"

    pulse_val = _fmt(synth["pulse_mean"])
    pulse_cls = _score_cls(synth["pulse_mean"])
    sol_val = _fmt(sol)
    sol_cls = _score_cls(sol)

    signals_html = "".join(
        f"<div class='signal-card'><h4>{title}</h4><p>{detail}</p></div>"
        for title, detail in synth["combined_signals"]
    )

    priority_title, priority_detail = synth["top_priority"]

    return HTML.format(
        team=team,
        n=n_respondents,
        wb_face=wb_face, wb_label=wb_label,
        acc_face=acc_face,
        acc_val=_fmt(access), acc_col=acc_col, acc_arrow=acc_arrow,
        load_face=load_face,
        load_val=_fmt(cog), load_col=load_col, load_arrow=load_arrow,
        wb_pct_chip=wb_pct_chip,
        acc_pct_chip=acc_pct_chip,
        load_pct_chip=load_pct_chip,
        quote_text=quote_text,
        quote_attrib=quote_attrib,
        strengths_html=strengths_html, gaps_html=gaps_html,
        mood_chart=mood_chart, stress_chart=stress_chart,
        gauge=gauge, gauge_lbl=gauge_lbl, gauge_col=gauge_col,
        teamwork_html=teamwork_html, load_html=load_html,
        solution_html=solution_html,
        disc_quadrant=disc_quadrant,
        disc_dom_letter=disc_dom_letter,
        disc_dom_name=disc_dom_name,
        disc_dom_colour=disc_dom_colour,
        disc_commentary=disc["commentary"],
        disc_n=disc["n"],
        disc_diversity_pct=disc_diversity_pct,
        disc_cards_html=disc_cards_html,
        synth_headline=synth["headline"],
        synth_trajectory=synth["trajectory"],
        pulse_val=pulse_val,
        pulse_cls=pulse_cls,
        sol_val=sol_val,
        sol_cls=sol_cls,
        priority_title=priority_title,
        priority_detail=priority_detail,
        signals_html=signals_html,
        ai_summary_block=ai_summary_block,
    )


def _fmt(v: float) -> str:
    return "\u2014" if pd.isna(v) else f"{v:.0f}"


def _trend_arrow(current: float, prior: float,
                 higher_is_worse: bool = False) -> str:
    """Small coloured arrow string. Threshold: 3 points."""
    if pd.isna(current) or pd.isna(prior):
        return "\u25a0"  # filled square = no trend info
    delta = current - prior
    if abs(delta) < 3:
        return "\u25a0"
    worse = (delta > 0) if higher_is_worse else (delta < 0)
    return "\u25b2" if not worse else "\u25bc"  # up/down triangle


def _pct_change(current: float, prior: float,
                higher_is_worse: bool = False) -> tuple[str, str]:
    """Return (text, colour) for a percentage-change label.

    Uses relative percent change: (new - old) / old * 100. For cognitive
    load where higher_is_worse=True, a positive percent is bad (red).
    If either value is missing or the prior is zero, returns ("", "") so
    the caller can hide the label entirely.
    """
    if pd.isna(current) or pd.isna(prior) or prior == 0:
        return ("", "")
    pct = (current - prior) / prior * 100.0
    # Below ~3% we treat it as noise - suppress rather than show "+1%"
    if abs(pct) < 3:
        return (f"{pct:+.0f}%", "#94a3b8")
    worse = (pct > 0) if higher_is_worse else (pct < 0)
    colour = "#ef4444" if worse else "#22c55e"
    return (f"{pct:+.0f}%", colour)


def _per_individual_series(ind1: pd.DataFrame, ind2: pd.DataFrame,
                           ind3: pd.DataFrame,
                           team: str, metric: str) -> list[float]:
    """Build a sequence S1 -> S2 -> S3 from individual-level readings.

    We concatenate Survey 1 individuals (ordered by Individual ID), then
    Survey 2, then Survey 3 - giving a true left-to-right timeline.
    """
    pts = []
    for ind in (ind1, ind2, ind3):
        if not ind.empty and metric in ind.columns:
            sub = ind[ind["Team"] == team].sort_values("Individual")
            pts.extend(sub[metric].tolist())
    # Trim to at most 12 points for readability
    if len(pts) > 12:
        idx = np.linspace(0, len(pts) - 1, 12).astype(int)
        pts = [pts[i] for i in idx]
    if len(pts) < 3:
        pts = pts + [np.nan] * (3 - len(pts))
    return pts


# --------------------------------------------------------------------------- #
# Main - the for-loop
# --------------------------------------------------------------------------- #

def main() -> None:
    s1 = add_team_columns(load_survey(SURVEY1))
    s2 = add_team_columns(load_survey(SURVEY2))
    s3 = add_team_columns(load_survey(SURVEY3))
    s4 = add_team_columns(load_survey(SURVEY4))

    # Persist the augmented CSVs (satisfies 'create a new column' requirement)
    s1.to_csv(OUTPUT / "Survey_1_with_TeamCols.csv", index=False)
    s2.to_csv(OUTPUT / "Survey_2_with_TeamCols.csv", index=False)
    s3.to_csv(OUTPUT / "Survey_3_with_TeamCols.csv", index=False)
    s4.to_csv(OUTPUT / "Survey_4_with_TeamCols.csv", index=False)

    # Pre-compute all the aggregates ONCE, then loop over teams -------------
    core1 = team_means(s1, METRIC_NEEDLES)
    core2 = team_means(s2, METRIC_NEEDLES)
    core3 = team_means(s3, METRIC_NEEDLES)
    core4 = team_means(s4, METRIC_NEEDLES)
    challenge2 = team_means(s2, CHALLENGE_NEEDLES)
    challenge3 = team_means(s3, CHALLENGE_NEEDLES)
    challenge4 = team_means(s4, CHALLENGE_NEEDLES)
    ind1 = individual_series(s1, METRIC_NEEDLES)
    ind2 = individual_series(s2, METRIC_NEEDLES)
    ind3 = individual_series(s3, METRIC_NEEDLES)
    ind4 = individual_series(s4, METRIC_NEEDLES)

    teams = sorted(set(s1["Team"])
                   .union(s2["Team"])
                   .union(s3["Team"])
                   .union(s4["Team"]))

    # === THE FOR LOOP =======================================================
    if AI_SUMMARY_ENABLED:
        print(f"Generating dashboards with AI summaries via {OLLAMA_MODEL}...")
        print(f"(Set AI_SUMMARY_ENABLED=False to skip the LLM calls.)")
    summary_rows = []
    for i, team in enumerate(teams, 1):
        if AI_SUMMARY_ENABLED:
            print(f"  [{i:2d}/{len(teams)}] Team {team}...", end=" ", flush=True)
        t0 = time.time()
        html = build_dashboard(team, s1, s2, s3, s4,
                               core1, core2, core3, core4,
                               challenge2, challenge3, challenge4,
                               ind1, ind2, ind3, ind4)
        out_path = OUTPUT / f"team_{team}_pulse.html"
        out_path.write_text(html, encoding="utf-8")
        if AI_SUMMARY_ENABLED:
            print(f"done ({time.time() - t0:.1f}s)")

        # Summary row for the roll-up CSV
        latest = _latest_row(team, core1, core2, core3, core4)
        if team in challenge4.index:
            sol = solution_score(challenge4.loc[team])
        elif team in challenge3.index:
            sol = solution_score(challenge3.loc[team])
        elif team in challenge2.index:
            sol = solution_score(challenge2.loc[team])
        else:
            sol = np.nan

        disc_prof = team_disc_profile(s1, team)

        summary_rows.append({
            "Team": team,
            "N_Respondents": (
                s1[s1["Team"] == team]["Individual"].nunique()
                + s2[s2["Team"] == team]["Individual"].nunique()
                + s3[s3["Team"] == team]["Individual"].nunique()
                + s4[s4["Team"] == team]["Individual"].nunique()
            ),
            **{f"{m}": round(latest.get(m), 1) if pd.notna(latest.get(m)) else None
               for m in METRIC_NEEDLES},
            "SolutionScore": round(sol, 1) if pd.notna(sol) else None,
            "DISC_Dominant": disc_prof["dominant"],
            "DISC_Diversity": round(disc_prof["diversity"], 2),
            "DISC_D_pct": round(disc_prof["percents"]["D"], 0),
            "DISC_I_pct": round(disc_prof["percents"]["I"], 0),
            "DISC_S_pct": round(disc_prof["percents"]["S"], 0),
            "DISC_C_pct": round(disc_prof["percents"]["C"], 0),
        })

    summary = pd.DataFrame(summary_rows)
    summary.to_csv(OUTPUT / "team_pulse_summary.csv", index=False)

    print(f"Generated {len(teams)} Team Pulse dashboards in {OUTPUT}")
    print()
    print(summary.to_string(index=False))


if __name__ == "__main__":
    main()

Generating dashboards with AI summaries via llama3...
(Set AI_SUMMARY_ENABLED=False to skip the LLM calls.)
  [ 1/22] Team 01... done (35.6s)
  [ 2/22] Team 02... done (18.4s)
  [ 3/22] Team 03... done (18.7s)
  [ 4/22] Team 04... done (27.5s)
  [ 5/22] Team 05... done (25.5s)
  [ 6/22] Team 06... done (24.3s)
  [ 7/22] Team 11... done (19.5s)
  [ 8/22] Team 12... done (25.6s)
  [ 9/22] Team 13... done (24.6s)
  [10/22] Team 14... done (21.1s)
  [11/22] Team 16... done (20.2s)
  [12/22] Team 17... done (23.3s)
  [13/22] Team 18... done (18.3s)
  [14/22] Team 19... done (24.6s)
  [15/22] Team 22... done (20.4s)
  [16/22] Team 23... done (26.3s)
  [17/22] Team 24... done (22.8s)
  [18/22] Team 25... done (17.8s)
  [19/22] Team 26... done (23.8s)
  [20/22] Team 27... done (22.6s)
  [21/22] Team 34... done (23.8s)
  [22/22] Team 35... done (24.9s)
Generated 22 Team Pulse dashboards in output

Team  N_Respondents  Wellbeing  Access  Productivity  Capacity  Capability  Collaboration  Safety 